# 00 – Datos: descarga y calidad
**Lab #2 – Derivados Financieros** · Punto 0 (rúbrica R4 – 3 %)

Objetivo: descargar los insumos y documentar fuente, fecha de descarga, limpieza, tratamiento de dividendos y coherencia temporal.

## 0.1 Precios (CSCO, ADI, AMGN, ^NDX)
CSCO, ADI y el ^NDX salen de la descarga del 01/11/2026 que ya teníamos. AMGN entra en vez de PYPL (asesoría del 09/11) y se bajó aparte el 10/11/2026.
Que sean dos fechas de descarga no cambia los retornos. Yahoo ajusta el Adj Close multiplicando toda la historia por un mismo factor, entonces los niveles se pueden mover pero los retornos de la ventana quedan iguales.

In [9]:
import yfinance as yf                      # para bajar datos de Yahoo Finance
import pandas as pd                        # tablas
import numpy as np                         # cuentas
from pathlib import Path                   # rutas que sirven en Windows y Linux
from datetime import date                  # para anotar el día de la descarga

RAW = Path("../data/raw")                  # carpeta de datos crudos (no se modifican)
INICIO = "2016-10-30"                      # 10 años, lo máximo que permite el profe
FIN = "2026-10-31"                         # yfinance no incluye el 'end', así entra el 30/10
ACTUALIZAR = False                         # CSCO, ADI, ^NDX y ^TNX se quedan con la descarga del 01/11

# AMGN: la primera vez va en True; después se pasa a False y se lee el CSV
BAJAR_AMGN = False
if BAJAR_AMGN:
    a = yf.download("AMGN", start=INICIO, end=FIN, auto_adjust=False, actions=True, progress=False)
    if isinstance(a.columns, pd.MultiIndex):           # algunas versiones devuelven 2 niveles de columnas
        a.columns = a.columns.get_level_values(0)      # nos quedamos solo con el nombre del campo
    a[["Close", "Adj Close", "Dividends", "Stock Splits"]].to_csv(RAW / "amgn_diario.csv")
    (RAW / "amgn_fecha_descarga.txt").write_text(str(date.today()))

FECHA_DESCARGA = {"CSCO, ADI, ^NDX, ^TNX": "2026-11-01",
                  "AMGN": (RAW / "amgn_fecha_descarga.txt").read_text()}

def leer_precios(nombre):                  # lee un CSV de data/raw y lo deja numérico
    t = pd.read_csv(RAW / nombre, index_col=0)
    t = t[pd.to_datetime(t.index, errors="coerce").notna()]   # quita filas raras de encabezado
    t.index = pd.to_datetime(t.index)
    return t.astype(float)

amgn = leer_precios("amgn_diario.csv")
VIEJOS = ["CSCO", "ADI", "^NDX"]           # lo que usamos de la descarga vieja (PYPL queda por fuera)
ORDEN = ["CSCO", "ADI", "AMGN", "^NDX"]

def unir(archivo_viejo, campo):            # pega la columna de AMGN a la tabla vieja
    viejo = leer_precios(archivo_viejo)[VIEJOS]
    return viejo.join(amgn[campo].rename("AMGN"), how="outer")[ORDEN]   # 'outer' para que un desfase salga como NaN

close = unir("close_diario.csv", "Close")
adj = unir("adjclose_diario.csv", "Adj Close")
divs = unir("dividendos.csv", "Dividends")

print("Fechas de descarga:", FECHA_DESCARGA)
print("Columnas:", list(adj.columns))
print("Rango:", adj.index.min().date(), "→", adj.index.max().date(), "|", len(adj), "días")
print("Splits de AMGN en la ventana:", int((amgn["Stock Splits"] > 0).sum()))

Fechas de descarga: {'CSCO, ADI, ^NDX, ^TNX': '2026-11-01', 'AMGN': '2026-11-10'}
Columnas: ['CSCO', 'ADI', 'AMGN', '^NDX']
Rango: 2016-10-31 → 2026-10-30 | 2514 días
Splits de AMGN en la ventana: 0


## 0.2 Tasa libre de riesgo (^TNX al 30/10/2026)

In [10]:
if ACTUALIZAR or not (RAW / "tnx.csv").exists():
    tnx = yf.download("^TNX", start="2026-10-01", end=FIN, auto_adjust=False, progress=False)
    tnx.to_csv(RAW / "tnx.csv")                            # serie cruda de ^TNX
tnx = pd.read_csv(RAW / "tnx.csv", header=[0, 1], index_col=0)    # columnas en dos niveles (campo, ticker)
tnx = tnx[pd.to_datetime(tnx.index, errors="coerce").notna()]     # quita la fila "Date" vacía
tnx.index = pd.to_datetime(tnx.index)
rf_pct = float(tnx["Close"].squeeze().astype(float).loc["2026-10-30"])   # ^TNX cotiza en puntos %
print(f"^TNX al 30/10/2026: {rf_pct:.3f} % → r_f = {rf_pct/100:.5f}")

^TNX al 30/10/2026: 4.929 % → r_f = 0.04929


## 0.3 Controles de calidad

In [11]:
print("Datos de la descarga del:", FECHA_DESCARGA)
print("Rango:", adj.index.min().date(), "→", adj.index.max().date())  # debe terminar el 2026-10-30
print("Observaciones:", len(adj))
print("=" * 30)
print("NaN por activo:\n", adj.isna().sum())

Datos de la descarga del: {'CSCO, ADI, ^NDX, ^TNX': '2026-11-01', 'AMGN': '2026-11-10'}
Rango: 2016-10-31 → 2026-10-30
Observaciones: 2514
NaN por activo:
 CSCO    0
ADI     0
AMGN    0
^NDX    0
dtype: int64


## 0.4 Limpieza y dividendos
Revisamos que no haya fechas repetidas, precios raros ni huecos, miramos los movimientos más fuertes y sacamos los dividendos de cada acción.
También probamos que el Adj Close de Yahoo sí tenga bien metidos los dividendos, porque de eso depende todo lo que sigue.

In [12]:
# 1) Que las series estén sanas
print("Fechas repetidas:", close.index.duplicated().sum())
print("Fechas en orden:", close.index.is_monotonic_increasing)
print("Precios en cero o negativos:", (close <= 0).sum().to_dict())
print("Hueco más largo entre fechas (días):", close.index.to_series().diff().dt.days.max(),
      "→ fines de semana y festivos de la bolsa gringa")

# 2) Los movimientos diarios más fuertes (no se borran, son cosas que pasaron de verdad)
r_abs = np.log(close).diff().abs()
extremos = pd.DataFrame({"Mayor movimiento diario": r_abs.max(), "Fecha": r_abs.idxmax().dt.date})
display(extremos.style.format({"Mayor movimiento diario": "{:.2%}"}))

# 3) Dividendos de las 3 acciones
k = 12                                           # todo el lab va en mensual
c_m = close.resample("ME").last()                # cierre del último día de bolsa de cada mes
a_m = adj.resample("ME").last()
ACC = ["CSCO", "ADI", "AMGN"]
ult12 = divs.loc["2025-10-31":, ACC]             # lo que pagaron en el último año
tabla_div = pd.DataFrame({
    "Pagos en 10 años": (divs[ACC] > 0).sum(),
    "Pagos últimos 12 m": (ult12 > 0).sum(),     # deberían ser 4 (pagan cada trimestre)
    "Dividendos últimos 12 m (USD)": ult12.sum(),
    "q (rend. dividendo)": ult12.sum() / close[ACC].iloc[-1],
    "Ret. anual precio (Close)": np.exp(np.log(c_m[ACC]).diff().mean() * k) - 1,   # con retornos mensuales
    "Ret. anual total (Adj Close)": np.exp(np.log(a_m[ACC]).diff().mean() * k) - 1,
})
tabla_div["Aporte dividendos (pp)"] = tabla_div["Ret. anual total (Adj Close)"] - tabla_div["Ret. anual precio (Close)"]
display(tabla_div.style.format({"Dividendos últimos 12 m (USD)": "{:,.2f}", "q (rend. dividendo)": "{:.2%}",
        "Ret. anual precio (Close)": "{:.2%}", "Ret. anual total (Adj Close)": "{:.2%}",
        "Aporte dividendos (pp)": "{:.2%}"}))

# 4) Prueba del Adj Close: en cada ex-dividendo el factor de Yahoo tiene que ser 1 − D / Close del día anterior
f = adj[ACC] / close[ACC]                        # factor de ajuste que usó Yahoo, día a día
dias_ex = divs[ACC] > 0                          # días en que hubo ex-dividendo
factor_yahoo = (f.shift(1) / f)[dias_ex]         # cuánto cambió el factor ese día
factor_teorico = (1 - divs[ACC] / close[ACC].shift(1))[dias_ex]
error_pb = (factor_yahoo - factor_teorico).abs().max() * 1e4   # diferencia máxima en puntos básicos
print("Máxima diferencia en el factor de ajuste (pb):", error_pb.round(3).to_dict())

print("^NDX: Adj Close = Close en toda la serie →", np.allclose(adj["^NDX"], close["^NDX"]),
      "(es índice de precio, no mete dividendos)")

Fechas repetidas: 0
Fechas en orden: True
Precios en cero o negativos: {'CSCO': 0, 'ADI': 0, 'AMGN': 0, '^NDX': 0}
Hueco más largo entre fechas (días): 4.0 → fines de semana y festivos de la bolsa gringa


,Mayor movimiento diario,Fecha
CSCO,14.77%,2022-05-19
ADI,18.17%,2020-03-16
AMGN,11.17%,2024-05-03
^NDX,13.00%,2020-03-16


,Pagos en 10 años,Pagos últimos 12 m,Dividendos últimos 12 m (USD),q (rend. dividendo),Ret. anual precio (Close),Ret. anual total (Adj Close),Aporte dividendos (pp)
CSCO,40,4,1.67,1.60%,13.04%,16.40%,3.36%
ADI,40,4,4.29,1.03%,20.60%,22.84%,2.25%
AMGN,40,4,9.94,2.13%,12.69%,16.10%,3.41%


Máxima diferencia en el factor de ajuste (pb): {'CSCO': 0.003, 'ADI': 0.005, 'AMGN': 0.003}
^NDX: Adj Close = Close en toda la serie → True (es índice de precio, no mete dividendos)


In [13]:
# Guardamos las series ya revisadas para que 01 y 02 lean exactamente esto (y no los CSV crudos con PYPL)
PROC = Path("../data/processed")
for nombre, tabla in {"close_v2.csv": close, "adj_v2.csv": adj, "divs_v2.csv": divs}.items():
    tabla.to_csv(PROC / nombre)                  # columnas CSCO, ADI, AMGN, ^NDX
print("Guardado en data/processed:", "close_v2.csv, adj_v2.csv, divs_v2.csv")

Guardado en data/processed: close_v2.csv, adj_v2.csv, divs_v2.csv


## 0.5 Resumen de los datos

Todos los precios salen de Yahoo Finance (con yfinance): cierre diario normal (Close), cierre ajustado (Adj Close) y dividendos.
CSCO, ADI, el ^NDX y la ^TNX son de la descarga del 01/11/2026. AMGN se bajó el 10/11/2026 porque entró en vez de PYPL después de la asesoría.
Que sean dos fechas no cambia los retornos: Yahoo ajusta toda la historia por el mismo factor, y de hecho CSCO y ADI dan exactamente lo mismo que en la primera versión.

La ventana va del 31/10/2016 al 30/10/2026 (10 años, 2.514 días de bolsa). No hay datos faltantes, fechas repetidas ni precios raros.

Trabajamos con Adj Close, o sea retorno total con dividendos incluidos. Las tres acciones pagan cada trimestre (40 pagos en 10 años).
En el último año: CSCO rinde 1,60 % en dividendos, ADI 1,03 % y AMGN 2,13 %. Probamos que Yahoo sí metió bien esos dividendos en el Adj Close (la diferencia da menos de 0,01 pb).
Para no contarlos dos veces en el punto 11, separamos el retorno en precio + dividendos.
El ^NDX es índice de precio (no incluye dividendos), y eso importa para el precio teórico del futuro.

Usamos retornos logarítmicos mensuales (k = 12) en todo el laboratorio: estadística, Markowitz, VaR, betas y simulación. Lo diario solo aparece como referencia.
La tasa libre de riesgo es la ^TNX al 30/10/2026: 4,929 % anual.


Al final guardamos close, adj y divs en `data/processed/` (`close_v2.csv`, `adj_v2.csv`, `divs_v2.csv`). Los notebooks 01 y 02 leen esos archivos, así todo el lab usa los mismos datos.